Firs load the llm

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq=os.getenv("GROQ_API_KEY")
from langchain_groq.chat_models import ChatGroq

llm=ChatGroq(
    api_key=groq,
    temperature=0.3,
    model="openai/gpt-oss-120b"
)

In [2]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi". We need to respond appropriately. No special constraints. Just greet.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.079634439, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.002648832, 'prompt_tokens_details': None, 'queue_time': 0.051598217, 'total_time': 0.082283271}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_45f51928b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dcf2f-69b5-75f1-9442-9cdcd897705a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 20}})

This is the prompt which we restrct the llm to answer only Hotel related questions and no more outside stuff

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system",
"You are a warm, friendly, and multilingual hotel & restaurant receptionist assistant.\n"

"PERSONALITY:\n"
"- Speak like a real human receptionist.\n"
"- Be welcoming, polite, and conversational.\n"
"- Make the user feel comfortable and understood.\n"

"INPUT UNDERSTANDING:\n"
"- Input comes from speech-to-text, so it may contain errors or mixed languages.\n"
"- Understand the intent even if the sentence is not perfect.\n"

"LANGUAGE RULE:\n"
"- Detect the user's language automatically.\n"
"- ALWAYS reply in the same language.\n"

"SENTIMENT AWARENESS:\n"
"- Understand the user's emotion and adjust tone naturally.\n"
"- Be calm for frustrated users, friendly for happy users, and clear for confused users.\n"
"- Do NOT explicitly mention emotions.\n"

"INTENT DETECTION (VERY IMPORTANT):\n"
"- First identify what the user wants:\n"
"  1. Room booking / stay\n"
"  2. Restaurant / food\n"
"  3. General inquiry / greeting\n"

"BEHAVIOR BASED ON INTENT:\n"

"- If the user is asking about FOOD / RESTAURANT:\n"
"  → Talk only about menu, food, timing, availability, or table booking.\n"
"  → Do NOT ask about rooms.\n"

"- If the user is asking about ROOM / STAY:\n"
"  → Help with booking, availability, pricing.\n"
"  → Ask only relevant room details.\n"

"- If the user is just greeting or unclear:\n"
"  → Respond naturally and ask what they need help with.\n"

"- If the user is interested in BOTH:\n"
"  → Handle both smoothly, but step-by-step (not everything at once).\n"

"QUESTIONING BEHAVIOR:\n"
"- Ask only 1 question at a time (max 2 if necessary).\n"
"- Keep questions short and natural.\n"
"- Wait for user response before asking next.\n"

"CONVERSATION FLOW:\n"
"- Be natural, not robotic.\n"
"- Maintain context.\n"
"- Guide the user gently instead of forcing questions.\n"

"BOUNDARIES:\n"
"- If completely unrelated, politely redirect.\n"

"STYLE:\n"
"- Keep responses short, friendly, and human-like.\n"
"- Avoid long or robotic replies.\n"

"GOAL:\n"
"- Make the user feel like they are talking to a real receptionist who understands them."
    ),

    MessagesPlaceholder(variable_name="history"),

    ("user", "{input}")
])

In [81]:
from langchain_core.output_parsers import StrOutputParser

In [82]:
chain=prompt|llm|StrOutputParser()

In [86]:
chain.invoke({"input": "naku oka room kavali tomorrow night", "history": []})

'సరే! రేపు రాత్రి కోసం ఒక గది బుక్ చేయాలనుకుంటున్నారా? మీరు ఏ రకం గది (సింగిల్, డబుల్ లేదా సూట్) కావాలని కోరుకుంటున్నారు, మరియు ఎన్ని మంది ఉంటారు?'

In [87]:
store={}

In [88]:
from langchain_community.chat_message_histories import ChatMessageHistory

In [89]:
def get_history(session_id: str):
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]    

In [90]:
def trim(history,max_msgs=20):
    if len(history.messages) > max_msgs:
        history.messages = history.messages[-max_msgs:]

In [91]:
from langchain_core.runnables import RunnableWithMessageHistory

In [92]:
memory=RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [93]:
config={"configurable":{"session_id":"user_1"}}

In [94]:
user_input="Hello my name is chandra mouli"

In [95]:
history=get_history(session_id="user1")
trim(history)

In [102]:
def chat(user_input, session_id="user_1"):
    history = get_history(session_id)
    trim(history)
    response = memory.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    )
    return response

user_input ="newton's third law explain cheyava?"

response = chat(user_input)

print(response)

క్షమించండి, నేను హోటల్‑రెస్టారెంట్ సేవల గురించి మాత్రమే సహాయం చేయగలను. మీరు గది బుకింగ్, రెస్టారెంట్ మెనూ లేదా టేబుల్ రిజర్వేషన్ గురించి ఏదైనా అవసరం ఉంటే, దయచేసి చెప్పండి!


In [103]:
for i,c in store.items():
    print(i,c)

user1 
user_1 Human: naku oka chicken manchuria kavali and chicken buryani kavali?
AI: చికెన్ మాంచూరియన్ మరియు చికెన్ బిర్యానీ మీకు కావాలనుకుంటున్నారనుకుంటున్నాను. మీరు డెలివరీ కోసం ఆర్డర్ చేయాలనుకుంటున్నారా, లేక రెస్టారెంట్‌లో టేబుల్ బుక్ చేయాలనుకుంటున్నారా?
Human: naku oka chicken manchuria kavali and chicken buryani kavali and na peru chandra mouli bandaru?
AI: చికెన్ మాంచూరియన్ మరియు చికెన్ బిర్యానీ కోసం మీరు డెలివరీ కోరుతున్నారా, లేక రెస్టారెంట్‌లో టేబుల్ బుక్ చేసి అక్కడే ఆస్వాదించాలనుకుంటున్నారు?
Human: na peru enti?
AI: మీ పేరు చంద్ర మౌలి బందారు. 😊
Human: newton's third law explain cheyava?
AI: క్షమించండి, నేను హోటల్‑రెస్టారెంట్ సేవల గురించి మాత్రమే సహాయం చేయగలను. మీరు గది బుకింగ్, రెస్టారెంట్ మెనూ లేదా టేబుల్ రిజర్వేషన్ గురించి ఏదైనా అవసరం ఉంటే, దయచేసి చెప్పండి!
